# COLMAP Reconstruction

To use COLMAP dense reconstruction only, I downloaded the binaries from the [website](https://github.com/colmap/colmap/releases) and ran the reconstruction.

To use it, I opened COLMAP using the executable, and selected the Automatic Reconstruction option, setting the folders (input from the "images" folder and output the "reconstructions/COLMAP" folder). I have enabled the GPU usage.

In [2]:
import sys
sys.path.append("../")
from utils.pointCloud import visualize_point_cloud

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Visualizing Point Cloud

In [4]:
#visualize_point_cloud("../reconstructions/COLMAP_segmented/dense/0/fused.ply")
visualize_point_cloud("../reconstructions/colmap_20_False.ply")

[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The reques

## Converting to Mesh

In [ ]:
import open3d as o3d
import numpy as np

# Read the point cloud
pcd = o3d.io.read_point_cloud("../reconstructions/colmap_20_False.ply")

# Estimate normals (required for surface reconstruction)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
)

# Orient normals consistently
pcd.orient_normals_consistent_tangent_plane(k=15)

# Surface Reconstruction (often better quality)
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=15
)
vertices_to_remove = densities < np.quantile(densities, 0.01)
mesh.remove_vertices_by_mask(vertices_to_remove)

# Save the mesh
o3d.io.write_triangle_mesh("../reconstructions/mesh_colmap.ply", mesh)

# Visualize the result
o3d.visualization.draw_geometries([mesh])

## Notes

It used 20 images to run, working really well (8/10). It just misses some parts on the miniature, making some blank spaces. The down part of it is relying on the GPU to run the dense reconstruction. and the time taken (> 20min).

It worked using the segmented images, but the missing parts were higher and had black points around the miniature